# 44. 프로젝트 B — 데이터 분석 Agent

> **제44장** · **이론편 대응: 25장 종합**
> **예상 소요**: 120분
> **필요 사양**: **[CPU]** 로 실행 가능
> **선행 장**: **42번을 먼저 실행** (utils 모듈 생성)
> **API 키**: 선택 (없어도 도구·루프 전부 동작)

---

## 만들 것

**데이터를 받아 스스로 계산하고 분석해 보고하는 Agent**

```
목표 → 계획 → 도구 실행 → 결과 확인 → 반복 → 보고
```

| 단계 | 절 | 쓰는 기술 |
|---|---|---|
| 1 | 분석할 데이터 준비 | — |
| 2 | **도구 만들기** | 37장 2절 |
| 3 | 안전장치 | 37장 7절 |
| 4 | **ReAct 루프** ★ | 37장 5절 |
| 5 | 평가 — 성공률 측정 | 27장 8절 |
| 6 | **실패 분석과 개선** ★ | 37장 8절 |
| 7 | 보고서 생성 | 24장 7절 |
| 8 | 비용과 지표 | 31번 |

**43장과 무엇이 다른가**

| | 43번 (문서 QA) | 44번 (분석 Agent) |
|---|---|---|
| 핵심 질문 | 무엇을 아는가 | **무엇을 할 수 있는가** |
| 모델 호출 | 1~2회 | **여러 회 반복** |
| 실패 지점 | 검색이 틀림 | **도구 실행 오류** |
| 검증 대상 | 근거가 맞는가 | **결론이 맞는가** |

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if root.name.startswith("part"):
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

try:
    from utils import set_seed, setup_korean_font
    setup_korean_font()
    set_seed(42)
    print("utils 모듈 로드 완료 (42장에서 생성)")
except ImportError:
    print("[주의] utils 모듈이 없습니다. 42장을 먼저 실행하세요.")
    import matplotlib.pyplot as plt
    import matplotlib.font_manager as fm
    import platform
    _c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
          "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
    _a = {f.name for f in fm.fontManager.ttflist}
    for _n in _c.get(platform.system(), []):
        if _n in _a:
            plt.rcParams["font.family"] = _n
            break
    plt.rcParams["axes.unicode_minus"] = False

import numpy as np
import matplotlib.pyplot as plt
import json
import time
import os
import ast
import operator
import inspect

try:
    from dotenv import load_dotenv
    load_dotenv(root / ".env")
except ImportError:
    pass

API_KEY, BASE_URL, MODEL = None, None, "gpt-4o-mini"
for env_name, base, model in [
        ("OPENAI_API_KEY", None, "gpt-4o-mini"),
        ("GROQ_API_KEY", "https://api.groq.com/openai/v1", "llama-3.3-70b-versatile"),
        ("GEMINI_API_KEY", "https://generativelanguage.googleapis.com/v1beta/openai/", "gemini-2.0-flash")]:
    if os.getenv(env_name):
        API_KEY, BASE_URL, MODEL = os.getenv(env_name), base, model
        print(f"API 키: {env_name} ({MODEL})")
        break
if not API_KEY:
    print("API 키 없음 — 도구와 루프 구조는 전부 동작합니다")

---

## 1. 분석할 데이터 준비

**매출 데이터**를 만든다. 외부 다운로드 없이 코드로 생성하되,
**분석할 거리가 있도록** 지역·월별 경향을 심어 둔다.

In [ ]:
import numpy as np
import json

np.random.seed(42)

REGIONS = ["서울", "부산", "대구", "광주"]
MONTHS = [f"2026-{m:02d}" for m in range(1, 7)]

# 지역별 기본 규모와 성장률을 다르게 설정
REGION_CONFIG = {
    "서울": {"base": 5000, "growth": 0.08},
    "부산": {"base": 3000, "growth": 0.05},
    "대구": {"base": 2000, "growth": -0.03},   # 감소 추세
    "광주": {"base": 1500, "growth": 0.12},    # 높은 성장
}

sales_data = []
for i, month in enumerate(MONTHS):
    for region in REGIONS:
        cfg = REGION_CONFIG[region]
        trend = cfg["base"] * (1 + cfg["growth"]) ** i
        noise = np.random.normal(0, cfg["base"] * 0.05)
        amount = max(0, int(trend + noise))
        sales_data.append({
            "month": month,
            "region": region,
            "amount": amount,
            "orders": int(amount / np.random.uniform(45, 55)),
        })

print("=" * 78)
print("분석 대상 데이터")
print("=" * 78)
print(f"기간: {MONTHS[0]} ~ {MONTHS[-1]} ({len(MONTHS)}개월)")
print(f"지역: {', '.join(REGIONS)}")
print(f"레코드: {len(sales_data)}건")
print()
print("샘플")
for row in sales_data[:4]:
    print(f"  {row}")
print("  ...")
print()

total = sum(r["amount"] for r in sales_data)
print(f"총 매출: {total:,}")
print(f"월 평균: {total/len(MONTHS):,.0f}")
print()
print("[데이터에 심어 둔 것]")
print("  서울: 규모 최대, 꾸준한 성장")
print("  대구: 감소 추세 (문제 지역)")
print("  광주: 규모는 작지만 성장률 최고")
print()
print("  Agent 가 이것들을 찾아내는지 보는 것이 목표다.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

# 지역별 추이
ax = axes[0]
colors = {"서울": "#1E40AF", "부산": "#0D9488",
          "대구": "#DC2626", "광주": "#EA580C"}
for region in REGIONS:
    amounts = [r["amount"] for r in sales_data if r["region"] == region]
    ax.plot(MONTHS, amounts, marker="o", linewidth=2,
            color=colors[region], label=region)
ax.set_xlabel("월")
ax.set_ylabel("매출")
ax.set_title("지역별 매출 추이")
ax.legend()
ax.grid(alpha=0.3)
ax.tick_params(axis="x", rotation=45, labelsize=8)

# 지역별 합계
ax = axes[1]
totals = {r: sum(x["amount"] for x in sales_data if x["region"] == r)
          for r in REGIONS}
bars = ax.bar(list(totals.keys()), list(totals.values()),
              color=[colors[r] for r in totals])
for b, v in zip(bars, totals.values()):
    ax.text(b.get_x()+b.get_width()/2, v+400, f"{v:,}", ha="center", fontsize=8)
ax.set_ylabel("총 매출")
ax.set_title("지역별 합계")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("사람은 이 그래프를 보면 바로 안다.")
print("  Agent 는 숫자만 보고 이것을 알아내야 한다.")

---

## 2. 도구 만들기 — 37장 2절

Agent가 쓸 도구를 만든다. **각 도구는 하나만 잘한다**는 원칙을 따른다.

In [ ]:
import numpy as np
import json


# ── 도구 1: 데이터 조회 ──
def query_sales(region: str = "all", month: str = "all") -> str:
    """매출 데이터를 조회합니다. region과 month로 필터링할 수 있습니다."""
    rows = sales_data
    if region != "all":
        rows = [r for r in rows if r["region"] == region]
    if month != "all":
        rows = [r for r in rows if r["month"] == month]

    if not rows:
        return f"조건에 맞는 데이터가 없습니다 (region={region}, month={month})"

    total = sum(r["amount"] for r in rows)
    return json.dumps({
        "count": len(rows),
        "total_amount": total,
        "avg_amount": round(total / len(rows), 1),
        "rows": rows[:12],       # 너무 길면 잘라낸다 (37장 6절)
    }, ensure_ascii=False)


# ── 도구 2: 집계 ──
def aggregate_sales(group_by: str, metric: str = "amount") -> str:
    """매출을 그룹별로 집계합니다. group_by는 'region' 또는 'month'입니다."""
    if group_by not in ("region", "month"):
        return "오류: group_by는 'region' 또는 'month'여야 합니다"
    if metric not in ("amount", "orders"):
        return "오류: metric은 'amount' 또는 'orders'여야 합니다"

    groups = {}
    for r in sales_data:
        key = r[group_by]
        groups[key] = groups.get(key, 0) + r[metric]

    ordered = dict(sorted(groups.items(), key=lambda x: -x[1]))
    return json.dumps(ordered, ensure_ascii=False)


# ── 도구 3: 성장률 계산 ──
def growth_rate(region: str) -> str:
    """특정 지역의 첫 달 대비 마지막 달 성장률(%)을 계산합니다."""
    rows = [r for r in sales_data if r["region"] == region]
    if len(rows) < 2:
        return f"오류: '{region}' 데이터가 부족합니다"

    rows.sort(key=lambda r: r["month"])
    first, last = rows[0]["amount"], rows[-1]["amount"]
    if first == 0:
        return "오류: 첫 달 매출이 0입니다"

    rate = (last - first) / first * 100
    return json.dumps({
        "region": region,
        "first_month": rows[0]["month"], "first_amount": first,
        "last_month": rows[-1]["month"], "last_amount": last,
        "growth_rate_pct": round(rate, 2),
    }, ensure_ascii=False)


# ── 도구 4: 안전한 계산기 (37장 2절) ──
_SAFE_OPS = {ast.Add: operator.add, ast.Sub: operator.sub,
             ast.Mult: operator.mul, ast.Div: operator.truediv,
             ast.Pow: operator.pow, ast.Mod: operator.mod,
             ast.USub: operator.neg, ast.UAdd: operator.pos}


def _safe_eval(node):
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    if isinstance(node, ast.Constant):
        if not isinstance(node.value, (int, float)):
            raise ValueError("숫자만 허용됩니다")
        return node.value
    if isinstance(node, ast.BinOp):
        op = _SAFE_OPS.get(type(node.op))
        if op is None:
            raise ValueError("허용되지 않은 연산자")
        return op(_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp):
        op = _SAFE_OPS.get(type(node.op))
        if op is None:
            raise ValueError("허용되지 않은 연산자")
        return op(_safe_eval(node.operand))
    raise ValueError(f"허용되지 않은 표현식: {type(node).__name__}")


def calculator(expression: str) -> str:
    """수식을 계산합니다. 사칙연산과 거듭제곱을 지원합니다."""
    try:
        return str(round(_safe_eval(ast.parse(expression, mode="eval")), 4))
    except Exception as e:
        return f"계산 오류: {e}"


# ── 도구 5: 통계 ──
def statistics(region: str = "all") -> str:
    """매출의 기초 통계(평균, 최대, 최소, 표준편차)를 구합니다."""
    rows = sales_data if region == "all" else [
        r for r in sales_data if r["region"] == region]
    if not rows:
        return f"오류: '{region}' 데이터가 없습니다"

    amounts = np.array([r["amount"] for r in rows])
    return json.dumps({
        "region": region, "n": len(amounts),
        "mean": round(float(amounts.mean()), 1),
        "std": round(float(amounts.std()), 1),
        "min": int(amounts.min()), "max": int(amounts.max()),
    }, ensure_ascii=False)


TOOLS = {
    "query_sales": query_sales,
    "aggregate_sales": aggregate_sales,
    "growth_rate": growth_rate,
    "calculator": calculator,
    "statistics": statistics,
}

print("=" * 78)
print("도구 동작 확인")
print("=" * 78)
print(f"\naggregate_sales('region'):")
print(f"  {aggregate_sales('region')}")
print(f"\ngrowth_rate('대구'):")
print(f"  {growth_rate('대구')}")
print(f"\nstatistics('서울'):")
print(f"  {statistics('서울')}")
print(f"\ncalculator('5000 * 1.08 ** 5'):")
print(f"  {calculator('5000 * 1.08 ** 5')}")

In [ ]:
import inspect
import json


def make_schema(func):
    """함수에서 도구 스키마 생성 (37장 2절)"""
    sig = inspect.signature(func)
    type_map = {str: "string", int: "integer", float: "number", bool: "boolean"}
    props, required = {}, []
    for name, p in sig.parameters.items():
        props[name] = {"type": type_map.get(p.annotation, "string")}
        if p.default is inspect.Parameter.empty:
            required.append(name)
    return {
        "type": "function",
        "function": {
            "name": func.__name__,
            "description": (func.__doc__ or "").strip().split("\n")[0],
            "parameters": {"type": "object", "properties": props,
                           "required": required},
        },
    }


SCHEMAS = [make_schema(f) for f in TOOLS.values()]

print("=" * 78)
print("도구 스키마")
print("=" * 78)
for s in SCHEMAS:
    f = s["function"]
    params = ", ".join(f["parameters"]["properties"].keys())
    req = f["parameters"]["required"]
    print(f"\n  {f['name']}({params})")
    print(f"    {f['description']}")
    print(f"    필수 인자: {req if req else '없음'}")

print()
print("-" * 78)
schema_chars = len(json.dumps(SCHEMAS, ensure_ascii=False))
print(f"전체 스키마 크기: {schema_chars}자 (약 {schema_chars//3} 토큰)")
print("  매 호출마다 이만큼이 프롬프트에 들어간다 (37장 6절)")

---

## 3. 안전장치 — 37장 7절

**Agent는 실제로 실행한다.** 시작 전에 안전장치부터 확인한다.

In [ ]:
print("=" * 78)
print("안전 점검 1: 계산기")
print("=" * 78)
print()

dangerous = [
    "__import__('os').system('ls')",
    "open('/etc/passwd').read()",
    "[].__class__.__base__.__subclasses__()",
]

print("위험한 입력을 넣어 본다")
for d in dangerous:
    result = calculator(d)
    print(f"  {d[:42]:<44}{result[:30]}")

print()
print("정상 계산은 유지된다")
for expr in ["5000 * 1.08", "(6000 - 5000) / 5000 * 100"]:
    print(f"  {expr:<44}{calculator(expr)}")

print()
print("-" * 78)
print("안전 점검 2: 도구 인자 검증")
print("-" * 78)
print()

bad_calls = [
    ("aggregate_sales", {"group_by": "invalid"}),
    ("growth_rate", {"region": "존재하지않는지역"}),
    ("query_sales", {"region": "서울", "month": "2099-01"}),
]

for name, args in bad_calls:
    result = TOOLS[name](**args)
    print(f"  {name}({args})")
    print(f"    → {result[:60]}")

print()
print("[원칙] 오류를 예외로 던지지 않고 문자열로 돌려준다 (37장 8절)")
print("  Agent 가 오류 메시지를 읽고 스스로 고쳐 재시도할 수 있다.")

In [ ]:
import time


class SafetyGuard:
    # Agent 안전장치 (37장 7절)

    def __init__(self, max_steps=8, max_seconds=120, max_repeats=3):
        self.max_steps = max_steps
        self.max_seconds = max_seconds
        self.max_repeats = max_repeats
        self.reset()

    def reset(self):
        self.start_time = time.time()
        self.step_count = 0
        self.call_history = {}
        self.warnings = []

    def check_step(self):
        """다음 단계를 진행해도 되는지 확인"""
        self.step_count += 1

        if self.step_count > self.max_steps:
            self.warnings.append(f"최대 단계({self.max_steps}) 초과")
            return False, "최대 단계 수를 넘었습니다"

        elapsed = time.time() - self.start_time
        if elapsed > self.max_seconds:
            self.warnings.append(f"시간 초과 ({elapsed:.0f}초)")
            return False, "시간 제한을 넘었습니다"

        return True, None

    def check_call(self, tool_name, args):
        """같은 호출이 반복되는지 확인"""
        key = (tool_name, json.dumps(args, sort_keys=True, ensure_ascii=False))
        self.call_history[key] = self.call_history.get(key, 0) + 1

        if self.call_history[key] > self.max_repeats:
            msg = f"'{tool_name}' 같은 인자로 {self.max_repeats}회 초과 호출"
            self.warnings.append(msg)
            return False, msg
        return True, None

    def report(self):
        return {
            "steps": self.step_count,
            "elapsed": time.time() - self.start_time,
            "unique_calls": len(self.call_history),
            "total_calls": sum(self.call_history.values()),
            "warnings": self.warnings,
        }


print("=" * 78)
print("안전장치 동작 확인")
print("=" * 78)

guard = SafetyGuard(max_steps=3, max_repeats=2)

print("\n단계 제한 (max_steps=3)")
for i in range(5):
    ok, msg = guard.check_step()
    print(f"  {i+1}단계: {'진행' if ok else '중단 — ' + msg}")
    if not ok:
        break

print("\n반복 호출 감지 (max_repeats=2)")
guard.reset()
for i in range(4):
    ok, msg = guard.check_call("growth_rate", {"region": "서울"})
    print(f"  {i+1}회: {'허용' if ok else '차단 — ' + msg}")
    if not ok:
        break

print()
print("경고 기록:", guard.warnings)

---

## 4. ReAct 루프 ★ — 37장 5절

**Agent의 본체**를 만든다. 37장에서 만든 것에 안전장치와 기록을 더한다.

In [ ]:
import json
import time


def call_llm(messages, tools=None, max_tokens=800, temperature=0.0):
    """LLM 호출 (25장 방식)"""
    if not API_KEY:
        return None
    try:
        from openai import OpenAI
        kwargs = {"api_key": API_KEY}
        if BASE_URL:
            kwargs["base_url"] = BASE_URL
        client = OpenAI(**kwargs)
        params = {"model": MODEL, "messages": messages,
                  "max_tokens": max_tokens, "temperature": temperature}
        if tools:
            params["tools"] = tools
        return client.chat.completions.create(**params)
    except Exception as e:
        return {"_error": f"{type(e).__name__}: {str(e)[:150]}"}


class AnalysisAgent:
    # 데이터 분석 Agent (37장 5절 확장)

    SYSTEM_PROMPT = """당신은 매출 데이터를 분석하는 도우미입니다.

작업 방식:
1. 목표를 이루려면 어떤 정보가 필요한지 생각합니다.
2. 도구를 호출해 정보를 얻습니다.
3. 얻은 결과를 확인하고, 부족하면 다시 도구를 씁니다.
4. 충분해지면 결론을 정리해 답합니다.

주의사항:
- 계산은 반드시 도구를 사용하세요. 직접 암산하지 마세요.
- 도구 결과에 없는 숫자를 지어내지 마세요.
- 결론에는 근거가 된 수치를 함께 제시하세요."""

    def __init__(self, tools, schemas, guard=None, verbose=True):
        self.tools = tools
        self.schemas = schemas
        self.guard = guard or SafetyGuard()
        self.verbose = verbose

    def run_tool(self, name, args):
        """도구 실행 — 오류도 문자열로 돌려준다"""
        func = self.tools.get(name)
        if func is None:
            return f"오류: '{name}' 도구가 없습니다. " \
                   f"사용 가능: {list(self.tools.keys())}"
        try:
            return str(func(**args))
        except TypeError as e:
            return f"인자 오류: {e}"
        except Exception as e:
            return f"실행 오류: {type(e).__name__}: {e}"

    def run(self, goal):
        self.guard.reset()
        messages = [
            {"role": "system", "content": self.SYSTEM_PROMPT},
            {"role": "user", "content": goal},
        ]
        trace = []

        if self.verbose:
            print(f"목표: {goal}")
            print("-" * 78)

        while True:
            ok, msg = self.guard.check_step()
            if not ok:
                return self._finish(goal, None, trace, stopped=msg)

            response = call_llm(messages, tools=self.schemas)

            if response is None:
                return self._finish(goal, None, trace, stopped="API 키 없음")
            if isinstance(response, dict) and "_error" in response:
                return self._finish(goal, None, trace, stopped=response["_error"])

            message = response.choices[0].message

            # 도구 호출이 없으면 최종 답변
            if not message.tool_calls:
                return self._finish(goal, message.content, trace)

            messages.append(message.model_dump())

            for tc in message.tool_calls:
                name = tc.function.name
                try:
                    args = json.loads(tc.function.arguments)
                except json.JSONDecodeError:
                    args = {}

                allowed, warn = self.guard.check_call(name, args)
                if not allowed:
                    output = f"중단: {warn}. 다른 방법을 시도하세요."
                else:
                    output = self.run_tool(name, args)

                if self.verbose:
                    print(f"[{self.guard.step_count}단계] {name}({args})")
                    print(f"        → {output[:70]}")

                trace.append({"step": self.guard.step_count, "tool": name,
                              "args": args, "result": output})
                messages.append({"role": "tool", "tool_call_id": tc.id,
                                 "content": output})

    def _finish(self, goal, answer, trace, stopped=None):
        report = self.guard.report()
        result = {"goal": goal, "answer": answer, "trace": trace,
                  "stopped": stopped, **report}
        if self.verbose:
            print("-" * 78)
            if answer:
                print("결론")
                for line in answer.strip().split("\n"):
                    print(f"  {line}")
            elif stopped:
                print(f"[중단] {stopped}")
            print()
            print(f"단계 {report['steps']}, "
                  f"도구 호출 {report['total_calls']}회, "
                  f"{report['elapsed']:.1f}초")
        return result


agent = AnalysisAgent(TOOLS, SCHEMAS, SafetyGuard(max_steps=8))

print("=" * 78)
print("Agent 실행")
print("=" * 78)
result = agent.run("어느 지역의 매출이 가장 높은가요? 총액도 알려주세요.")

In [ ]:
print("=" * 78)
print("여러 목표 실행")
print("=" * 78)

goals = [
    "매출이 감소하고 있는 지역이 있나요?",
    "광주와 대구의 성장률을 비교해 주세요.",
]

results = []
for g in goals:
    print(f"\n{'='*78}")
    r = agent.run(g)
    results.append(r)

if not API_KEY:
    print()
    print("=" * 78)
    print("API 키가 없어 실제 루프는 돌지 않았습니다.")
    print("도구를 직접 호출해 Agent 가 할 일을 확인해 봅니다.")
    print("=" * 78)
    print()
    print("목표: '매출이 감소하고 있는 지역이 있나요?'")
    print()
    print("  Agent 가 밟을 단계")
    for i, region in enumerate(REGIONS, 1):
        result = growth_rate(region)
        data = json.loads(result)
        trend = "감소" if data["growth_rate_pct"] < 0 else "증가"
        print(f"  {i}단계 growth_rate('{region}')")
        print(f"        → {data['growth_rate_pct']:+.2f}% ({trend})")
    print()
    print("  결론: 대구가 감소 추세입니다.")

---

## 5. 평가 — 성공률 측정 — 27장 8절

43장에서 검색 성능을 측정했듯, **Agent도 측정해야 한다.**

다만 측정이 더 어렵다. "정답 문서"가 아니라 **"결론이 맞는가"**를 봐야 하기 때문이다.

In [ ]:
import json
import re


# 평가 데이터 — 목표와 정답
agent_eval_set = [
    {
        "goal": "어느 지역의 매출이 가장 높은가요?",
        "check": lambda ans: "서울" in ans,
        "expected": "서울",
        "difficulty": "쉬움",
    },
    {
        "goal": "매출이 감소하는 지역이 있나요?",
        "check": lambda ans: "대구" in ans,
        "expected": "대구",
        "difficulty": "보통",
    },
    {
        "goal": "성장률이 가장 높은 지역은?",
        "check": lambda ans: "광주" in ans,
        "expected": "광주",
        "difficulty": "보통",
    },
    {
        "goal": "서울의 6개월 총 매출은 얼마인가요?",
        "check": lambda ans: any(
            str(x) in ans.replace(",", "")
            for x in [sum(r["amount"] for r in sales_data if r["region"] == "서울")]),
        "expected": str(sum(r["amount"] for r in sales_data if r["region"] == "서울")),
        "difficulty": "보통",
    },
    {
        "goal": "전체 매출에서 서울이 차지하는 비중은 몇 %인가요?",
        "check": lambda ans: any(
            f"{p}" in ans for p in
            [str(round(sum(r["amount"] for r in sales_data if r["region"]=="서울")
                       / sum(r["amount"] for r in sales_data) * 100, i))
             for i in range(3)]),
        "expected": f"{sum(r['amount'] for r in sales_data if r['region']=='서울')/sum(r['amount'] for r in sales_data)*100:.1f}%",
        "difficulty": "어려움",
    },
]

print("=" * 78)
print("Agent 평가 데이터")
print("=" * 78)
print(f"{'목표':<40}{'기대 답':<16}{'난이도'}")
print("-" * 78)
for e in agent_eval_set:
    print(f"{e['goal'][:38]:<40}{e['expected'][:14]:<16}{e['difficulty']}")
print("-" * 78)
print()
print("[Agent 평가가 어려운 이유]")
print()
print("  43장의 검색은 '정답 문서 ID'가 명확했다.")
print("  Agent 는 자유로운 문장으로 답하므로 정답 판정이 애매하다.")
print()
print("  여기서는 핵심 키워드나 숫자가 답변에 있는지로 판정한다.")
print("  실무에서는 사람이 검토하거나 LLM 으로 채점하기도 한다.")

In [ ]:
import time
import numpy as np


def evaluate_agent(agent, eval_set, verbose=True):
    """Agent 성능 평가"""
    records = []

    for e in eval_set:
        agent.verbose = False
        t0 = time.time()
        result = agent.run(e["goal"])
        elapsed = time.time() - t0

        answer = result.get("answer") or ""
        success = bool(answer) and e["check"](answer)

        records.append({
            "goal": e["goal"],
            "difficulty": e["difficulty"],
            "success": success,
            "steps": result["steps"],
            "tool_calls": result["total_calls"],
            "elapsed": elapsed,
            "stopped": result.get("stopped"),
            "answer": answer[:80] if answer else None,
        })

        if verbose:
            mark = "성공" if success else ("중단" if result.get("stopped") else "실패")
            print(f"  [{mark}] {e['goal'][:40]:<42}"
                  f"{result['steps']}단계 {elapsed:.1f}초")

    agent.verbose = True
    return records


print("=" * 78)
print("Agent 평가 실행")
print("=" * 78)

if API_KEY:
    records = evaluate_agent(agent, agent_eval_set)

    print()
    print("-" * 78)
    n = len(records)
    success = sum(r["success"] for r in records)
    print(f"성공률      : {success}/{n} ({success/n:.1%})")
    print(f"평균 단계   : {np.mean([r['steps'] for r in records]):.1f}")
    print(f"평균 도구 호출: {np.mean([r['tool_calls'] for r in records]):.1f}회")
    print(f"평균 소요   : {np.mean([r['elapsed'] for r in records]):.1f}초")
    print("-" * 78)

    print()
    print("난이도별 성공률")
    for diff in ["쉬움", "보통", "어려움"]:
        subset = [r for r in records if r["difficulty"] == diff]
        if subset:
            s = sum(r["success"] for r in subset)
            print(f"  {diff:<10}{s}/{len(subset)}")
else:
    records = []
    print("  (API 키가 없어 평가를 건너뜁니다)")
    print()
    print("  평가에서 측정하는 것")
    print(f"    {'성공률':<16}결론이 맞았는가")
    print(f"    {'평균 단계':<16}몇 번 반복했는가 (적을수록 효율적)")
    print(f"    {'도구 호출 수':<16}비용과 직결")
    print(f"    {'소요 시간':<16}사용자 대기 시간")

---

## 6. 실패 분석과 개선 ★ — 37장 8절

**측정했으면 개선한다.** 43장 5절에서 검색을 개선했듯, Agent도 같은 절차를 따른다.

In [ ]:
print("=" * 78)
print("Agent 가 실패하는 유형 (37장 8절)")
print("=" * 78)
print()
print(f"{'유형':<22}{'증상':<30}{'개선 방법'}")
print("-" * 78)
patterns = [
    ("도구 미사용",   "계산이 필요한데 암산으로 답함",   "프롬프트에서 강조"),
    ("잘못된 인자",   "형식에 맞지 않는 값 전달",       "스키마 설명 개선"),
    ("결과 무시",     "도구 결과와 다른 결론",         "근거 제시 요구"),
    ("과도한 반복",   "같은 도구를 여러 번",           "안전장치 (3절)"),
    ("중도 포기",     "충분히 알아보지 않고 답함",      "단계별 지시 추가"),
    ("도구 부족",     "필요한 도구가 없음",           "도구 추가"),
]
for a, b, c in patterns:
    print(f"{a:<22}{b:<30}{c}")
print("-" * 78)
print()

# 실제 실패 사례 분석
if records:
    failures = [r for r in records if not r["success"]]
    if failures:
        print(f"이번 평가의 실패 사례: {len(failures)}건")
        for f in failures:
            print(f"\n  목표: {f['goal']}")
            print(f"  답변: {f['answer']}")
            print(f"  단계: {f['steps']}, 도구 호출: {f['tool_calls']}회")
            if f["stopped"]:
                print(f"  중단 사유: {f['stopped']}")
    else:
        print("이번 평가에서는 모두 성공했습니다.")
else:
    print("(API 키가 없어 실제 실패 사례는 없습니다)")

In [ ]:
# 개선된 시스템 프롬프트
IMPROVED_PROMPT = """당신은 매출 데이터를 분석하는 도우미입니다.

작업 절차:
1. 목표를 이루려면 어떤 정보가 필요한지 나열합니다.
2. 필요한 정보를 도구로 하나씩 얻습니다.
3. 여러 대상을 비교해야 하면 **모두** 조회한 뒤 비교합니다.
4. 얻은 수치로 결론을 내립니다.

반드시 지킬 것:
- 계산은 예외 없이 도구를 사용합니다. 암산하지 마세요.
- 도구가 돌려준 숫자만 사용하고, 없는 숫자를 만들지 마세요.
- "가장 높은/낮은"을 묻는 질문은 **모든 대상을 조회한 뒤** 답하세요.
- 결론에는 근거 수치를 함께 적으세요.
- 도구 실행에 실패하면 인자를 고쳐 다시 시도하세요.

사용 가능한 지역: 서울, 부산, 대구, 광주
사용 가능한 기간: 2026-01 ~ 2026-06"""


class ImprovedAgent(AnalysisAgent):
    SYSTEM_PROMPT = IMPROVED_PROMPT


print("=" * 78)
print("프롬프트 개선")
print("=" * 78)
print()
print("무엇을 바꿨나")
print()
changes = [
    ("절차를 번호로 명시",      "무엇부터 할지 헷갈리지 않게"),
    ("'모두 조회한 뒤 비교'",   "일부만 보고 결론 내는 것 방지"),
    ("사용 가능한 값 명시",     "잘못된 인자 전달 방지"),
    ("실패 시 재시도 지시",     "오류를 만나도 포기하지 않게"),
]
for a, b in changes:
    print(f"  {a:<28}{b}")

print()
print(f"프롬프트 길이: {len(AnalysisAgent.SYSTEM_PROMPT)}자 → {len(IMPROVED_PROMPT)}자")
print()

improved = ImprovedAgent(TOOLS, SCHEMAS, SafetyGuard(max_steps=10))

if API_KEY:
    print("=" * 78)
    print("개선 후 재평가")
    print("=" * 78)
    records_v2 = evaluate_agent(improved, agent_eval_set)

    print()
    print("-" * 78)
    n = len(records_v2)
    s1 = sum(r["success"] for r in records)
    s2 = sum(r["success"] for r in records_v2)
    print(f"{'지표':<20}{'기존':<16}{'개선 후':<16}{'변화'}")
    print("-" * 78)
    print(f"{'성공률':<20}{s1}/{n} ({s1/n:.0%})    {s2}/{n} ({s2/n:.0%})    {s2-s1:+d}")
    print(f"{'평균 단계':<20}{np.mean([r['steps'] for r in records]):<16.1f}"
          f"{np.mean([r['steps'] for r in records_v2]):<16.1f}"
          f"{np.mean([r['steps'] for r in records_v2])-np.mean([r['steps'] for r in records]):+.1f}")
    print("-" * 78)
else:
    records_v2 = []
    print("(API 키가 없어 재평가를 건너뜁니다)")
    print()
    print("43장 5절에서 검색을 개선했듯, Agent 도 같은 절차를 따른다:")
    print("  측정 → 실패 분석 → 개선 → 재측정")

---

## 7. 보고서 생성 — 24장 7절

Agent가 분석한 결과를 **구조화된 형태로** 받는다.

In [ ]:
import json


REPORT_PROMPT = """당신은 매출 분석 보고서를 작성합니다.

도구로 필요한 데이터를 모두 조회한 뒤, 아래 JSON 형식으로만 답하세요.
다른 설명은 덧붙이지 마세요.

{
  "summary": "한 문장 요약",
  "total_sales": 숫자,
  "top_region": "지역명",
  "declining_regions": ["지역명"],
  "fastest_growing": "지역명",
  "findings": ["발견한 사실 1", "발견한 사실 2"],
  "recommendations": ["제안 1", "제안 2"]
}"""


class ReportAgent(ImprovedAgent):
    SYSTEM_PROMPT = REPORT_PROMPT


def parse_json_response(text):
    """모델 응답에서 JSON 추출 (24장 7절)"""
    if not text:
        return None, "빈 응답"
    cleaned = text.strip().replace("```json", "").replace("```", "").strip()
    start, end = cleaned.find("{"), cleaned.rfind("}")
    if start != -1 and end != -1:
        cleaned = cleaned[start:end+1]
    try:
        return json.loads(cleaned), None
    except json.JSONDecodeError as e:
        return None, str(e)


print("=" * 78)
print("구조화된 보고서 생성")
print("=" * 78)

report_agent = ReportAgent(TOOLS, SCHEMAS, SafetyGuard(max_steps=12))
result = report_agent.run(
    "2026년 상반기 매출을 분석해 보고서를 작성해 주세요.")

if result.get("answer"):
    data, err = parse_json_response(result["answer"])
    if data:
        print()
        print("파싱 성공 — 프로그램에서 바로 쓸 수 있다")
        print(json.dumps(data, ensure_ascii=False, indent=2))
    else:
        print(f"\n파싱 실패: {err}")
        print(f"원본: {result['answer'][:200]}")
else:
    print()
    print("(API 키 없음 — 도구로 직접 계산한 결과를 보여줍니다)")
    print()

    totals = json.loads(aggregate_sales("region"))
    growths = {r: json.loads(growth_rate(r))["growth_rate_pct"] for r in REGIONS}
    total_all = sum(totals.values())

    manual_report = {
        "summary": f"상반기 총 매출 {total_all:,}, 서울이 최대 규모",
        "total_sales": total_all,
        "top_region": max(totals, key=totals.get),
        "declining_regions": [r for r, g in growths.items() if g < 0],
        "fastest_growing": max(growths, key=growths.get),
        "findings": [
            f"{max(totals, key=totals.get)}가 전체의 "
            f"{max(totals.values())/total_all*100:.1f}% 차지",
            f"{max(growths, key=growths.get)}의 성장률이 "
            f"{max(growths.values()):.1f}%로 최고",
        ],
        "recommendations": [
            "감소 지역의 원인 분석 필요",
            "고성장 지역에 자원 배분 검토",
        ],
    }
    print(json.dumps(manual_report, ensure_ascii=False, indent=2))

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

print("=" * 78)
print("Agent 결과 검증")
print("=" * 78)
print()
print("Agent 가 낸 결론이 실제 데이터와 맞는지 직접 확인한다.")
print("  35장 7절에서 다룬 '모델의 말을 그대로 믿지 않는다'는 원칙이다.")
print()

# 실제 값 계산
actual_totals = {r: sum(x["amount"] for x in sales_data if x["region"] == r)
                 for r in REGIONS}
actual_growths = {r: json.loads(growth_rate(r))["growth_rate_pct"]
                  for r in REGIONS}

print(f"{'지역':<10}{'총 매출':<16}{'성장률':<14}{'판정'}")
print("-" * 78)
for r in REGIONS:
    trend = "감소" if actual_growths[r] < 0 else "증가"
    print(f"{r:<10}{actual_totals[r]:<16,}{actual_growths[r]:>+8.2f}%     {trend}")
print("-" * 78)
print()
print(f"매출 1위     : {max(actual_totals, key=actual_totals.get)}")
print(f"성장률 1위   : {max(actual_growths, key=actual_growths.get)}")
print(f"감소 지역    : {[r for r, g in actual_growths.items() if g < 0]}")
print()
print("1절에서 데이터에 심어 둔 것과 일치하는지 확인해 보자.")
print("  서울: 규모 최대  /  대구: 감소  /  광주: 성장률 최고")

---

## 8. 비용과 지표 — 41번

Agent는 여러 번 호출하므로 **비용 관리가 특히 중요하다.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("=" * 78)
print("Agent 비용 구조")
print("=" * 78)
print()

# 토큰 추정
schema_tokens = len(json.dumps(SCHEMAS, ensure_ascii=False)) // 3
system_tokens = len(IMPROVED_PROMPT) // 3
goal_tokens = 30
tool_result_tokens = 150
answer_tokens = 200

print(f"고정 비용 (매 호출마다)")
print(f"  시스템 프롬프트: 약 {system_tokens} 토큰")
print(f"  도구 스키마    : 약 {schema_tokens} 토큰")
print(f"  합계          : 약 {system_tokens + schema_tokens} 토큰")
print()

print("단계가 늘어날 때 누적 (37장 6절)")
print(f"{'단계':<8}{'이번 입력':<14}{'누적 입력':<14}{'배수'}")
print("-" * 78)

base = system_tokens + schema_tokens + goal_tokens
context = base
total_input = 0
history = []

for step in range(1, 9):
    total_input += context
    history.append(total_input)
    if step in (1, 2, 3, 5, 8):
        print(f"{step:<8}{context:<14,}{total_input:<14,}{total_input/base:.1f}배")
    context += 80 + tool_result_tokens      # 도구 요청 + 결과

print("-" * 78)
print()
print(f"8단계면 입력 토큰이 {total_input:,}개 — 단일 호출의 {total_input/base:.0f}배")
print()
print("[비용을 줄이는 방법]")
print("  1) 불필요한 도구는 스키마에서 제외 — 스키마도 매번 들어간다")
print("  2) 도구 결과 길이 제한 (2절의 rows[:12])")
print("  3) max_steps 제한 (3절)")
print("  4) 오래된 도구 결과 요약")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, 9), history, marker="o", linewidth=2, color="#EA580C")
ax.set_xlabel("Agent 단계")
ax.set_ylabel("누적 입력 토큰")
ax.set_title("단계에 따른 토큰 누적")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

print("=" * 78)
print("두 프로젝트 비교")
print("=" * 78)
print()
print(f"{'항목':<22}{'A: 문서 QA (43장)':<28}{'B: 분석 Agent (44장)'}")
print("-" * 78)
rows = [
    ("모델 호출 횟수",   "1~2회",                    "3~10회"),
    ("입력 토큰",       "프롬프트 + 문서 3개",        "누적 (단계마다 증가)"),
    ("지연",           "1~3초",                    "10~60초"),
    ("주요 실패",       "검색이 엉뚱한 문서",         "도구 오류, 중도 포기"),
    ("측정 방법",       "Recall, MRR",             "성공률, 단계 수"),
    ("품질 판정",       "정답 문서 ID 비교",         "결론 키워드/수치 확인"),
    ("개선 방법",       "검색 알고리즘",             "프롬프트, 도구 설계"),
]
for a, b, c in rows:
    print(f"{a:<22}{b:<28}{c}")
print("-" * 78)
print()
print("[언제 무엇을 쓸까]")
print()
print(f"{'상황':<40}{'권장'}")
print("-" * 78)
choices = [
    ("문서에 답이 있는 질문",              "A (RAG)"),
    ("계산이나 처리가 필요한 작업",         "B (Agent)"),
    ("빠른 응답이 중요",                  "A"),
    ("여러 단계를 거쳐야 하는 분석",        "B"),
    ("근거를 정확히 보여줘야 함",           "A"),
    ("외부 시스템과 연동 필요",            "B"),
    ("둘 다 필요",                       "Agent 안에 검색 도구를 넣는다"),
]
for a, b in choices:
    print(f"{a:<40}{b}")
print("-" * 78)
print()
print("마지막 항목이 실무에서 가장 흔하다.")
print("  43장의 검색기를 37번 Agent 의 도구 하나로 넣으면 된다.")

---

## 9. 개선 과제

이 Agent도 부족한 점이 많다. **직접 개선해 보자.**

| 난이도 | 과제 | 참고 |
|---|---|---|
| 쉬움 | 평가 목표를 10개 이상으로 늘리기 | 5절 |
| 쉬움 | 도구를 추가하고 성공률 재측정 | 2절 |
| 보통 | 43장의 검색기를 도구로 추가 | 43번 |
| 보통 | 시각화 도구 추가 (그래프 생성) | 03번 |
| 보통 | 프롬프트를 여러 버전으로 비교 | 6절 |
| 어려움 | 계획 수립 단계를 별도로 분리 | 38장 7절 |
| 어려움 | 여러 Agent로 역할 분담 | 30번 6~7절 |
| 어려움 | 결과를 검증하는 Agent 추가 | 38장 7절 |

**모든 개선마다 5절의 평가를 다시 돌리는 것**이 핵심이다.

In [ ]:
print("=" * 78)
print("과제 예시: 검색 도구 추가하기")
print("=" * 78)
print()
print("43장에서 만든 검색기를 이 Agent 의 도구로 넣으면")
print("데이터 분석과 규정 조회를 함께 하는 Agent 가 된다.")
print()
print("코드 형태")
print()
example = [
    "# 43장의 검색기를 불러온다",
    "from your_module import expanded_searcher",
    "",
    "def search_policy(keyword: str) -> str:",
    "    # 사내 규정을 검색합니다.",
    "    results = expanded_searcher.search(keyword, top_k=2)",
    "    return json.dumps([",
    "        {'source': r['source'], 'text': r['text'][:200]}",
    "        for r in results", 
    "    ], ensure_ascii=False)",
    "",
    "# 도구 목록에 추가",
    "TOOLS['search_policy'] = search_policy",
    "SCHEMAS.append(make_schema(search_policy))",
]
for line in example:
    print("  " + line)

print()
print("-" * 78)
print("이렇게 하면 이런 질문에 답할 수 있다")
print()
print("  '대구 지점 담당자가 서울로 출장 가면 비용이 얼마나 드나요?'")
print("    → search_policy('출장비') 로 규정 확인")
print("    → calculator 로 계산")
print()
print("  두 프로젝트가 하나로 합쳐지는 지점이다.")

---

## 10. 정리

### 만든 것

```
목표 → [계획 → 도구 실행 → 결과 확인] 반복 → 결론
              ↑                    ↓
           안전장치            실행 기록
```

### 이 프로젝트에서 배운 것

| 교훈 | 어디서 |
|---|---|
| 계산은 도구에 맡긴다 | 2절 — 암산하면 틀린다 |
| 안전장치가 먼저다 | 3절 — 실행하기 전에 |
| 오류는 예외 대신 메시지로 | 3절 — 스스로 고치게 |
| **Agent도 측정한다** | 5절 — 성공률, 단계 수 |
| 프롬프트가 성능을 좌우한다 | 6절 |
| 결론을 검증한다 | 7절 |
| 비용은 단계에 비례해 는다 | 8절 |

### 쓴 기술

| 장 | 어디에 |
|---|---|
| 37번 | 도구 호출, ReAct 루프, 안전장치 |
| 35번 | CoT (프롬프트의 절차 지시) |
| 24번 | 구조화된 출력 (JSON 보고서) |
| 25번 | API 호출, 오류 처리 |
| 41번 | 비용·지연 측정 |
| 38번 | 도구 표준화 개념 |

---

## 두 프로젝트를 마치며

**43장과 44장은 같은 재료로 다른 것을 만들었다.**

| | A: 문서 QA | B: 분석 Agent |
|---|---|---|
| 묻는 것 | "무엇을 아는가" | "무엇을 할 수 있는가" |
| 핵심 | 검색 품질 | 도구 설계 |
| 반복 | 없음 | 있음 |
| 위험 | 잘못된 근거 | 잘못된 실행 |

**공통점이 더 중요하다.**

1. **먼저 측정한다** — 평가 데이터 없이는 개선할 수 없다
2. **모든 개선이 효과 있는 것은 아니다** — 43장 5절에서 확인
3. **모델의 말을 그대로 믿지 않는다** — 근거 표시, 결과 검증
4. **안전장치는 나중이 아니라 먼저** — 되돌릴 수 없는 일이 있다

### 이 책을 마치며

실습 19개를 지나왔다. 되돌아보면 방식이 일정했다.

> **실패를 겪고 → 원인을 알고 → 직접 만들고 → 값을 확인한다**

이론편에서 손으로 계산한 값들이 코드에서 그대로 나온다는 것을 14번 확인했다.
이론과 구현이 같은 것을 말하고 있다는 뜻이다.

**앞으로 새로운 기술이 나와도** 이 방식은 유효하다.
무엇이 문제였고, 어떻게 풀었고, 그 대가가 무엇인지 —
이 세 가지를 묻는 습관이 이 책이 남기려 한 것이다.